# Neural Networks with TensorFlow (Keras)

## What you'll learn
- What TensorFlow and Keras are, and how they relate
- Building the same network as lesson 8 with `keras.Sequential`
- `compile`, `fit`, `evaluate`, and `predict`: training without writing the loop
- Using a validation split and early stopping to avoid overfitting
- Saving and loading a Keras model

TensorFlow is installed with `pip install tensorflow` (it is in `requirements.txt`).

## TensorFlow and Keras

**TensorFlow** is Google's neural network library. **Keras** is its high-level interface
and the usual way to use it: you describe the layers, pick a loss and an optimiser, and
call `fit`. Where PyTorch has you write the training loop (lesson 8), Keras runs it for you.
Both are widely used; the ideas are identical.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # hide TensorFlow's start-up messages

import keras
import tensorflow as tf

keras.utils.set_random_seed(42)   # repeatable results
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values
print(df.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]
X = pd.get_dummies(df[NUMERIC + ["segment", "region"]], columns=["segment", "region"], dtype=int)
y = df["paid_late"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Neural networks need scaled inputs. Fit the scaler on training data only.
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train).astype("float32")
X_test_s = scaler.transform(X_test).astype("float32")
print("inputs per account:", X_train_s.shape[1])

## Build the network

The same shape as lesson 8: two hidden layers and one output. Here the output layer uses a
**sigmoid** so it returns a probability directly.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(X_train_s.shape[1],)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.summary()

## Compile and fit

`compile` chooses the optimiser, the loss, and the metrics to report. `fit` runs the
training loop. `validation_split=0.2` holds back 20% of the **training** data to check
progress on unseen rows after every epoch.

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss="binary_crossentropy",
              metrics=["accuracy", keras.metrics.AUC(name="auc")])

history = model.fit(X_train_s, y_train, epochs=40, batch_size=64,
                    validation_split=0.2, verbose=0)
print("final training loss:", round(history.history["loss"][-1], 4))
print("final validation loss:", round(history.history["val_loss"][-1], 4))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["loss"], label="training")
plt.plot(history.history["val_loss"], label="validation")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Keras training vs validation loss")
plt.legend()
plt.show()

If training loss keeps falling while validation loss starts rising, the network is
memorising the training data (overfitting).

## Early stopping

`EarlyStopping` watches the validation loss and stops training when it has not improved for
a few epochs (`patience`), then restores the best weights. It removes the guesswork over how
many epochs to train.

In [ ]:
keras.utils.set_random_seed(42)
model = keras.Sequential([
    keras.Input(shape=(X_train_s.shape[1],)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy", keras.metrics.AUC(name="auc")])

stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
history = model.fit(X_train_s, y_train, epochs=200, batch_size=64,
                    validation_split=0.2, callbacks=[stop], verbose=0)
print("stopped after", len(history.history["loss"]), "epochs")

## Evaluate and predict

In [ ]:
from sklearn.metrics import roc_auc_score

loss, accuracy, auc = model.evaluate(X_test_s, y_test, verbose=0)
print(f"Test accuracy {accuracy:.1%}, Keras AUC {auc:.3f}")

probability = model.predict(X_test_s, verbose=0).ravel()
print(f"scikit-learn ROC AUC on the same predictions: {roc_auc_score(y_test, probability):.3f}")
print("first five probabilities:", probability[:5].round(2))

## Save and load

Keras saves the whole model (layers, weights, and training settings) in one `.keras` file.

In [ ]:
os.makedirs("models", exist_ok=True)
model.save("models/late_payment_keras.keras")

reloaded = keras.models.load_model("models/late_payment_keras.keras")
print(reloaded.predict(X_test_s[:3], verbose=0).ravel().round(2))

## PyTorch or TensorFlow?

| | PyTorch | TensorFlow / Keras |
|---|---|---|
| Training loop | You write it (full control) | `fit` runs it for you |
| Style | Plain Python, easy to debug | High-level, less code |
| Common in | Research, most new AI models | Production systems, mobile and web deployment |

Either is a fine choice. Learn the ideas once (layers, loss, optimiser, epochs,
validation) and they carry across.

## Practice

1. Add a `keras.layers.Dropout(0.2)` after each hidden layer and compare the test AUC.
2. Change `patience` to 2 and then to 15. How many epochs does training run each time?
3. Try a learning rate of 0.01 and of 0.0001 (`keras.optimizers.Adam(learning_rate=...)`).
   Plot the loss curves for both.
4. Train the same network on the **regression** target `next_year_spend` (use a final
   `Dense(1)` with no activation, `loss="mse"`, and scale the target by dividing by 100,000).

In [ ]:
# Your practice code here

## Summary

- Keras is TensorFlow's high-level interface: describe layers, `compile`, then `fit`.
- A sigmoid output gives a probability; `binary_crossentropy` trains yes/no models.
- A validation split shows overfitting; `EarlyStopping` stops at the best epoch.
- `evaluate` scores the test set; `predict` returns probabilities.
- `model.save` and `keras.models.load_model` store and reload the whole model.

## Practice Solutions

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import keras
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]
X = pd.get_dummies(df[NUMERIC + ["segment", "region"]], columns=["segment", "region"], dtype=int)
y = df["paid_late"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Neural networks need scaled inputs. Fit the scaler on training data only.
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train).astype("float32")
X_test_s = scaler.transform(X_test).astype("float32")


def network(dropout=0.0, learning_rate=0.001, output_activation="sigmoid"):
    keras.utils.set_random_seed(42)
    layers = [keras.Input(shape=(X_train_s.shape[1],)), keras.layers.Dense(32, activation="relu")]
    if dropout:
        layers.append(keras.layers.Dropout(dropout))
    layers.append(keras.layers.Dense(16, activation="relu"))
    if dropout:
        layers.append(keras.layers.Dropout(dropout))
    layers.append(keras.layers.Dense(1, activation=output_activation))
    return keras.Sequential(layers), keras.optimizers.Adam(learning_rate=learning_rate)


def train_classifier(dropout=0.0, learning_rate=0.001, patience=5):
    model, optimizer = network(dropout, learning_rate)
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=[keras.metrics.AUC(name="auc")])
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)
    history = model.fit(X_train_s, y_train, epochs=200, batch_size=64, validation_split=0.2,
                        callbacks=[stop], verbose=0)
    return model, history


# 1.
for dropout in [0.0, 0.2]:
    model, _ = train_classifier(dropout=dropout)
    print("dropout", dropout, "test AUC", round(model.evaluate(X_test_s, y_test, verbose=0)[1], 3))

# 2.
for patience in [2, 15]:
    _, history = train_classifier(patience=patience)
    print("patience", patience, "->", len(history.history["loss"]), "epochs")

# 3.
for lr in [0.01, 0.0001]:
    _, history = train_classifier(learning_rate=lr, patience=200)
    plt.plot(history.history["val_loss"], label=f"learning rate {lr}")
plt.xlabel("epoch"); plt.ylabel("validation loss"); plt.legend(); plt.show()

# 4.
s_train = df.loc[X_train.index, "next_year_spend"].to_numpy() / 100_000
s_test = df.loc[X_test.index, "next_year_spend"].to_numpy() / 100_000
model, optimizer = network(output_activation=None)
model.compile(optimizer=optimizer, loss="mse")
model.fit(X_train_s, s_train, epochs=100, batch_size=64, validation_split=0.2, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
predicted = model.predict(X_test_s, verbose=0).ravel() * 100_000
print(f"MAE: {mean_absolute_error(s_test * 100_000, predicted):,.0f} USD")